In [ ]:
!conda install -y gdown
!pip install google-play-scraper
!pip install PySastrawi

In [ ]:
from google_play_scraper import Sort, reviews_all, reviews
import re
import pandas as pd
import numpy as np
import datetime as dt
import torch
import string
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.metrics import confusion_matrix, classification_report
# from google_play_scraper import Sort, reviews_all, reviews
import nltk
# import calendar
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from wordcloud import WordCloud
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('popular')
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
label = ['positif', 'netral', 'negatif']
# Load pre-trained model and tokenizer
model_name = "ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Example input text
input_text = "aku cinta indonesia"

# Tokenize input text
input_ids = tokenizer.encode(input_text, add_special_tokens=True, max_length=1024, truncation=True, return_tensors='pt').to(device)

# Perform inference
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Get predicted label
predicted_label = torch.argmax(logits, dim=1).item()
print("Predicted label:",label[predicted_label])

In [ ]:
import re
from nltk.corpus import stopwords

def text_preprocessing(text):
  """
  Fungsi untuk melakukan text preprocessing dengan langkah-langkah:
  1. Cleansing
  2. Case Folding
  3. Tokenizing
  4. Filtering

  Args:
      text (str): Teks yang ingin diproses.

  Returns:
      list: List of kata-kata yang telah diproses.
  """

  # 1. Cleansing
  text = re.sub(r"[^\w\s]", "", text) # Menghapus karakter non-alfabetis
  text = text.lower() # Case folding

  # 2. Tokenizing
  tokens = text.split()

  # 3. Filtering
  # Mengganti stopwords bahasa Inggris dengan bahasa Indonesia
  stop_words = set(stopwords.words("indonesian"))
  filtered_tokens = [token for token in tokens if token not in stop_words]

  return ' '.join(filtered_tokens)

# Contoh penggunaan
text = "Contoh teks untuk preprocessing. Teks ini berisi tanda baca, simbol, dan kata-kata yang tidak penting."

preprocessed_text = text_preprocessing(text)

print(preprocessed_text)


In [ ]:
import pandas as pd

data = pd.read_csv('https://docs.google.com/spreadsheets/d/1xsx3f4lJBswSe39yxzvl7_k6M1OTSnSrJt5lpmIEyDA/gviz/tq?tqx=out:csv')

In [ ]:
data.head()

In [ ]:
data['clean_review'] = [text_preprocessing(data['textDisplay'][i]) for i in range(len(df))]

In [ ]:
data.head()

In [ ]:
output=[]
for i, val in data.iterrows():
    input_ids = tokenizer.encode(val['clean_review'], add_special_tokens=True,  max_length=512, truncation=True,  return_tensors='pt').to(device)
    with torch.no_grad():
        try:
            outputs = model(input_ids)
        except:
            print(val['clean_review'])
            break
        logits = outputs.logits
    predicted_label = torch.argmax(logits, dim=1).item()
    output.append(label[predicted_label])

In [ ]:
data['label'] = output
data.head()

In [ ]:
data.to_csv('/content/dataset_yt_febri_fix_label.csv', index=False)